In [1]:
import os
import time
from pathlib import Path
from io import StringIO
import pandas as pd
import requests

In [2]:
BASE_URL = "https://fingertips.phe.org.uk/api"
CHILD_AREA_TYPE_ID = 66  # Sub-ICB level
INDICATOR_IDS = [92163, 92167] # 92163: Prescribing volume (per 1000 pts), 92167: Prescribing quality (% Broad-spectrum)
PARENT_AREA_TYPE_ID = 15 # England
PARENT_AREA_CODE = "E92000001"

OUT_DIR = Path("data_raw")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
def fetch_csv(endpoint, params):
    url = f"{BASE_URL}{endpoint}"
    for attempt in range(6):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 429: 
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return pd.read_csv(StringIO(r.text))
        except Exception as e:
            print(f"尝试第 {attempt+1} 次失败: {e}")
            time.sleep(2)
    return None

def main():
    data_params = {
        "indicator_ids": ",".join(map(str, INDICATOR_IDS)),
        "child_area_type_id": CHILD_AREA_TYPE_ID,
        "parent_area_type_id": PARENT_AREA_TYPE_ID,
        "parent_area_code": PARENT_AREA_CODE,
    }
    
    df_data = fetch_csv("/all_data/csv/by_indicator_id", data_params)
    
    if df_data is not None:
        file_path = OUT_DIR / "antibiotics_raw_data.csv"
        df_data.to_csv(file_path, index=False)
        print(f"data saved to: {file_path}")
        print(f" total record {len(df_data)}")
    else:
        print("failed")

if __name__ == "__main__":
    main()

🚀 开始拉取抗生素数据...
✅ 成功！数据已保存至: data_raw/antibiotics_raw_data.csv
📊 共抓取到 9240 行数据。
